# CK — Chidamber & Kemerer Java Metrics

[mauricioaniche/ck](https://github.com/mauricioaniche/ck) computes class-level and method-level OO metrics for Java source trees.

## Metrics produced
| Abbreviation | Full name |
|---|---|
| CBO | Coupling Between Objects |
| DIT | Depth of Inheritance Tree |
| NOC | Number of Children |
| RFC | Response for a Class |
| WMC | Weighted Method Complexity |
| LCOM | Lack of Cohesion of Methods |
| LOC | Lines of Code |
| NOM | Number of Methods |
| NOPM | Number of Public Methods |
| NOSI | Number of Static Invocations |
| … and more (method-level, field-level) |  |

## Tool: `ck.jar` (CLI)
```
java -jar ck.jar <project-path> <use-jars> <max-files> <field-method-metrics> <output-dir>
```

## 1. Configuration

In [1]:
from __future__ import annotations
import os, shutil, subprocess, json, csv, io, sys
from pathlib import Path

_base = Path(r"F:\java_metrics")
PROJECT_ROOT   = Path(os.environ.get("JAVA_PROJECT_ROOT", _base / "sample-java-app" / "src")).resolve()
CK_JAR         = Path(os.environ.get("CK_JAR",           _base / "tools" / "ck.jar")).resolve()
OUTPUT_DIR     = _base / "ck_out"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Resolve real java.exe (avoid Windows System32 stub)
def _find_java() -> str:
    if jh := os.environ.get("JAVA_HOME"):
        p = Path(jh) / "bin" / ("java.exe" if sys.platform == "win32" else "java")
        if p.is_file(): return str(p)
    for candidate in (shutil.which("java") or "").splitlines():
        if "System32" not in candidate: return candidate
    return shutil.which("java") or "java"
JAVA_EXE = _find_java()

USE_JARS          = "false"
MAX_FILES         = "0"
FIELD_METHOD_METRICS = "true"

assert CK_JAR.is_file(),      f"ck.jar not found: {CK_JAR}"
assert PROJECT_ROOT.is_dir(), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
print(f"JAVA_EXE     : {JAVA_EXE}")
print(f"CK_JAR       : {CK_JAR}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")

JAVA_EXE     : C:\Program Files\Java\jdk-21.0.10\bin\java.EXE
CK_JAR       : F:\java_metrics\tools\ck.jar
PROJECT_ROOT : F:\java_metrics\sample-java-app\src
OUTPUT_DIR   : F:\java_metrics\ck_out


## 2. List Java source files to be analysed

In [2]:
java_files = sorted(PROJECT_ROOT.rglob("*.java"))
print(f"Java files found: {len(java_files)}")
for f in java_files:
    print(" ", f.relative_to(PROJECT_ROOT))

Java files found: 1
  main\java\com\example\App.java


## 3. Run CK — raw stdout / stderr

In [3]:
# CK treats last arg as FILE PREFIX — append os.sep so files land inside OUTPUT_DIR
CK_PREFIX = str(OUTPUT_DIR) + os.sep
cmd = [
    JAVA_EXE, "-jar", str(CK_JAR),
    str(PROJECT_ROOT),
    USE_JARS,
    MAX_FILES,
    FIELD_METHOD_METRICS,
    CK_PREFIX,
]
print("Command:", " ".join(cmd))

proc = subprocess.run(cmd, capture_output=True, text=True)

print("\n=== STDOUT ===")
print(proc.stdout or "(empty)")
print("\n=== STDERR ===")
print(proc.stderr or "(empty)")
print("\nExit code:", proc.returncode)

Command: C:\Program Files\Java\jdk-21.0.10\bin\java.EXE -jar F:\java_metrics\tools\ck.jar F:\java_metrics\sample-java-app\src false 0 true F:\java_metrics\ck_out\



=== STDOUT ===
Metrics extracted!!!


=== STDERR ===
log4j:WARN No appenders could be found for logger (com.github.mauricioaniche.ck.CK).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Exit code: 0


## 4. Raw CSV output — class-level metrics

In [4]:
class_csv = OUTPUT_DIR / "class.csv"
if class_csv.is_file():
    raw = class_csv.read_text(encoding="utf-8", errors="replace")
    print("=== class.csv (raw) ===")
    print(raw)
else:
    print(f"class.csv not found in {OUTPUT_DIR}. Check stderr above.")

=== class.csv (raw) ===
file,class,type,cbo,cboModified,fanin,fanout,wmc,dit,noc,rfc,lcom,lcom*,tcc,lcc,totalMethodsQty,staticMethodsQty,publicMethodsQty,privateMethodsQty,protectedMethodsQty,defaultMethodsQty,visibleMethodsQty,abstractMethodsQty,finalMethodsQty,synchronizedMethodsQty,totalFieldsQty,staticFieldsQty,publicFieldsQty,privateFieldsQty,protectedFieldsQty,defaultFieldsQty,finalFieldsQty,synchronizedFieldsQty,nosi,loc,returnQty,loopQty,comparisonsQty,tryCatchQty,parenthesizedExpsQty,stringLiteralsQty,numbersQty,assignmentsQty,mathOperationsQty,variablesQty,maxNestedBlocksQty,anonymousClassesQty,innerClassesQty,lambdasQty,uniqueWordsQty,modifiers,logStatementsQty
F:\java_metrics\sample-java-app\src\main\java\com\example\App.java,com.example.App,class,0,0,0,0,1,1,0,1,0,0.0,NaN,NaN,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,1,0,0,0,0,0,0,0,0,11,1,0



## 5. Raw CSV output — method-level metrics

In [5]:
method_csv = OUTPUT_DIR / "method.csv"
if method_csv.is_file():
    raw = method_csv.read_text(encoding="utf-8", errors="replace")
    print("=== method.csv (raw) ===")
    print(raw)
else:
    print(f"method.csv not found — may require FIELD_METHOD_METRICS=true (current: {FIELD_METHOD_METRICS}).")

=== method.csv (raw) ===
file,class,method,constructor,line,cbo,cboModified,fanin,fanout,wmc,rfc,loc,returnsQty,variablesQty,parametersQty,methodsInvokedQty,methodsInvokedLocalQty,methodsInvokedIndirectLocalQty,loopQty,comparisonsQty,tryCatchQty,parenthesizedExpsQty,stringLiteralsQty,numbersQty,assignmentsQty,mathOperationsQty,maxNestedBlocksQty,anonymousClassesQty,innerClassesQty,lambdasQty,uniqueWordsQty,modifiers,logStatementsQty,hasJavaDoc
F:\java_metrics\sample-java-app\src\main\java\com\example\App.java,com.example.App,main/1[java.lang.String[]],false,4,0,0,0,0,1,1,3,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,10,9,0,false



## 6. Raw CSV output — field-level metrics

In [6]:
field_csv = OUTPUT_DIR / "field.csv"
if field_csv.is_file():
    raw = field_csv.read_text(encoding="utf-8", errors="replace")
    print("=== field.csv (raw) ===")
    print(raw)
else:
    print("field.csv not found.")

=== field.csv (raw) ===
file,class,method,variable,usage



## 7. All output files produced

In [7]:
print("Files in output directory:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

Files in output directory:
  class.csv  (853 bytes)
  field.csv  (34 bytes)
  method.csv  (606 bytes)
  variable.csv  (152 bytes)
